In [138]:
import os
from pathlib import Path
import numpy as np
from pynwb import read_nwb
from matplotlib import pyplot as plt

from CSE583_humanSayMonkeyDo.load_config import load_config, get_config_value, get_data_paths


def get_nwbs(primate='monkey', max_subjects=None):
    """Return absolute Path objects for all data directories.

    Args:
        primate (str): 'monkey' or 'human' to specify
        max_subjects (int, optional): Maximum number of subjects to return. Defaults to None.
    Returns:
        dict: Dictionary with keys 'monkey' and 'human' mapping to lists of Path objects.
    """
    assert primate in ['monkey', 'human'], "primate must be 'monkey' or 'human'"
    assert (max_subjects is None) or (isinstance(max_subjects, int) and max_subjects > 0), "max_subjects must be a positive integer or None"
    
    data_paths = get_data_paths()
    nwb = list(data_paths[primate].glob("**/*.nwb"))
    if max_subjects is not None:
        nwb = nwb[:max_subjects]
    return nwb

def get_pos_chunk(hdf_dataset, start_times, end_times):
    """Extract chunks of data from an HDF5 dataset based on start and end times.
    
    Args:
        hdf_dataset: HDF5 dataset object with time-indexed data.
        start_times (list): List of start times for each chunk.
        end_times (list): List of end times for each chunk.
    
    Returns:
        list: List of numpy arrays containing the extracted data chunks.
    """
    # Assertions
    assert hdf_dataset is not None, "hdf_dataset cannot be None"
    assert hasattr(hdf_dataset, 'timestamps'), "hdf_dataset must have a 'timestamps' attribute"
    assert hasattr(hdf_dataset, 'data'), "hdf_dataset must have a 'data' attribute"
    assert len(start_times) == len(end_times), "start_times and end_times must have the same length"
    
    # Convert to numpy arrays once
    start_times = np.asarray(start_times)
    end_times = np.asarray(end_times)
    
    # Sort by start times for efficient sequential access
    sort_idx = np.argsort(start_times)
    sorted_start_times = start_times[sort_idx]
    sorted_end_times = end_times[sort_idx]
    
    # Get timestamps once (avoid repeated attribute access)
    timestamps = hdf_dataset.timestamps[:]
    
    # Vectorized searchsorted for all start/end times at once
    start_indices = np.searchsorted(timestamps, sorted_start_times, side='left')
    end_indices = np.searchsorted(timestamps, sorted_end_times, side='right')
    
    # Extract chunks
    chunks = []
    for start_idx, end_idx in zip(start_indices, end_indices):
        if start_idx < end_idx:  # Only add non-empty chunks
            chunk = hdf_dataset.data[start_idx:end_idx, :]
            chunks.append(chunk)
        else:
            chunks.append(np.array([]))  # Empty chunk
    
    return chunks

def get_windowed_pos_chunk(hdf_dataset, center_times, window_size):
    """Extract windowed chunks of data from an HDF5 dataset based on center times and window size.

    Args:
        hdf_dataset: HDF5 dataset object with time-indexed data.
        center_times (list): List of center times for each chunk.
        window_size (list): Two floats specifying the window size before and after the center time.
    Returns:
        numpy.ndarray: Array containing the extracted data chunks.
    """
    assert len(window_size) == 2, "window_size must be a list of two floats"
    start_times = [ct - window_size[0] for ct in center_times]
    end_times = [ct + window_size[1] for ct in center_times]
    return get_chunk(hdf_dataset, start_times, end_times)


def get_chunk_spikes(list_units_spkts, start_times, end_times, return_format='list'):
    """Extract spike times from a list of units based on start and end times.
    
    Args:
        list_units_spkts: List of spike times for each unit (each element is array-like).
        start_times (list): List of start times for each chunk.
        end_times (list): List of end times for each chunk.
        return_format (str): 'list', 'dict', or 'ragged' for output format.
    
    Returns:
        If return_format='list': List of shape [n_chunks][n_units] containing spike arrays
        If return_format='dict': Dict with keys 'spikes', 'counts', 'chunk_times'
        If return_format='ragged': Ragged array structure with metadata
    """
    # Convert to numpy arrays for faster operations
    start_times = np.asarray(start_times)
    end_times = np.asarray(end_times)
    n_chunks = len(start_times)
    n_units = len(list_units_spkts)
    
    # Pre-convert all units to numpy arrays once
    units_array = [np.asarray(unit) for unit in list_units_spkts]
    
    # Pre-allocate result structure
    chunked_list = [[None for _ in range(n_units)] for _ in range(n_chunks)]
    
    # Vectorized approach: use searchsorted for each unit
    for unit_idx, spikes in enumerate(units_array):
        if len(spikes) == 0:
            # Handle empty units
            for chunk_idx in range(n_chunks):
                chunked_list[chunk_idx][unit_idx] = np.array([])
            continue
            
        # Find indices for all chunks at once using searchsorted
        start_indices = np.searchsorted(spikes, start_times, side='left')
        end_indices = np.searchsorted(spikes, end_times, side='right')
        
        # Extract spikes for each chunk
        for chunk_idx in range(n_chunks):
            start_idx = start_indices[chunk_idx]
            end_idx = end_indices[chunk_idx]
            chunked_list[chunk_idx][unit_idx] = spikes[start_idx:end_idx]
    
    if return_format == 'dict':
        return {
            'spikes': chunked_list,
            'counts': [[len(chunked_list[c][u]) for u in range(n_units)] 
                       for c in range(n_chunks)],
            'chunk_times': list(zip(start_times, end_times)),
            'n_units': n_units,
            'n_chunks': n_chunks
        }
    elif return_format == 'ragged':
        # Return as structured array with metadata
        return {
            'data': chunked_list,
            'spike_counts': np.array([[len(chunked_list[c][u]) for u in range(n_units)] 
                                      for c in range(n_chunks)]),
            'start_times': start_times,
            'end_times': end_times
        }
    else:
        return chunked_list


def get_chunk_spikes_binned(list_units_spkts, start_times, end_times, bin_size=0.001):
    """Extract and bin spike times into fixed-size bins for array output.
    
    Args:
        list_units_spkts: List of spike times for each unit.
        start_times (list): List of start times for each chunk.
        end_times (list): List of end times for each chunk.
        bin_size (float): Size of time bins in seconds.
    
    Returns:
        numpy.ndarray: 3D array of shape [n_chunks, n_units, n_bins] with spike counts
        dict: Metadata including bin_edges, actual_times, etc.
    """
    start_times = np.asarray(start_times)
    end_times = np.asarray(end_times)
    n_chunks = len(start_times)
    n_units = len(list_units_spkts)
    
    # Calculate number of bins for each chunk
    chunk_durations = end_times - start_times
    n_bins_per_chunk = np.ceil(chunk_durations / bin_size).astype(int)
    max_bins = n_bins_per_chunk.max()
    
    # Pre-allocate 3D array
    binned_spikes = np.zeros((n_chunks, n_units, max_bins), dtype=np.int32)
    
    # Convert units to numpy arrays
    units_array = [np.asarray(unit) for unit in list_units_spkts]
    
    # Bin spikes for each chunk and unit
    for chunk_idx, (start, end, n_bins) in enumerate(zip(start_times, end_times, n_bins_per_chunk)):
        bin_edges = np.linspace(start, end, n_bins + 1)
        
        for unit_idx, spikes in enumerate(units_array):
            # Extract spikes in this chunk
            mask = (spikes >= start) & (spikes <= end)
            chunk_spikes = spikes[mask]
            
            if len(chunk_spikes) > 0:
                # Bin the spikes
                counts, _ = np.histogram(chunk_spikes, bins=bin_edges)
                binned_spikes[chunk_idx, unit_idx, :n_bins] = counts
    
    metadata = {
        'bin_size': bin_size,
        'n_bins_per_chunk': n_bins_per_chunk,
        'start_times': start_times,
        'end_times': end_times,
        'chunk_durations': chunk_durations
    }
    
    return binned_spikes, metadata

def get_chunk_spikes_binned_windowed(list_units_spkts, center_times, window_size, bin_size=0.001):
    start_times = [ct - window_size[0] for ct in center_times]
    end_times = [ct + window_size[1] for ct in center_times]
    return get_chunk_spikes_binned(list_units_spkts, start_times, end_times, bin_size=bin_size)


def get_chunk_spikes_aligned(list_units_spkts, start_times, end_times, max_duration=None):
    """Extract spikes and pad to create aligned 3D array.
    
    Args:
        list_units_spkts: List of spike times for each unit.
        start_times (list): List of start times for each chunk.
        end_times (list): List of end times for each chunk.
        max_duration (float): Maximum duration to consider. If None, uses longest chunk.
    
    Returns:
        numpy.ndarray: 3D array [n_chunks, n_units, max_spikes] with spike times (relative to chunk start)
        numpy.ndarray: 3D array [n_chunks, n_units, max_spikes] with valid spike mask
        dict: Metadata
    """
    start_times = np.asarray(start_times)
    end_times = np.asarray(end_times)
    n_chunks = len(start_times)
    n_units = len(list_units_spkts)
    
    units_array = [np.asarray(unit) for unit in list_units_spkts]
    
    # First pass: find maximum number of spikes in any chunk for any unit
    max_spikes = 0
    spike_lists = [[None for _ in range(n_units)] for _ in range(n_chunks)]
    
    for unit_idx, spikes in enumerate(units_array):
        start_indices = np.searchsorted(spikes, start_times, side='left')
        end_indices = np.searchsorted(spikes, end_times, side='right')
        
        for chunk_idx in range(n_chunks):
            chunk_spikes = spikes[start_indices[chunk_idx]:end_indices[chunk_idx]]
            # Convert to relative times
            relative_spikes = chunk_spikes - start_times[chunk_idx]
            spike_lists[chunk_idx][unit_idx] = relative_spikes
            max_spikes = max(max_spikes, len(relative_spikes))
    
    # Second pass: create padded arrays
    spike_array = np.full((n_chunks, n_units, max_spikes), np.nan, dtype=np.float32)
    valid_mask = np.zeros((n_chunks, n_units, max_spikes), dtype=bool)
    
    for chunk_idx in range(n_chunks):
        for unit_idx in range(n_units):
            spikes = spike_lists[chunk_idx][unit_idx]
            n_spikes = len(spikes)
            if n_spikes > 0:
                spike_array[chunk_idx, unit_idx, :n_spikes] = spikes
                valid_mask[chunk_idx, unit_idx, :n_spikes] = True
    
    metadata = {
        'max_spikes': max_spikes,
        'start_times': start_times,
        'end_times': end_times,
        'spike_counts': valid_mask.sum(axis=2)  # [n_chunks, n_units]
    }
    
    return spike_array, valid_mask, metadata

In [77]:
#Get paths to monkey NWB files
monkey_paths = get_nwbs('monkey', 2)
#Load first NWB file
nwbfile = read_nwb(monkey_paths[0])
#Visualize entries
nwbfile


Data type,object
Shape,"(4,)"
Array size,32.00 bytes
Chunk shape,None
Compression,None
Compression opts,None
Uncompressed size (bytes),32
Compressed size (bytes),64
Compression ratio,0.5
Data type,float64
Shape,"(100516, 2)"


In [ ]:
print(nwbfile.trials[:].keys())
# Lets view the data stored in the NWB file
cursor_pos = nwbfile.processing['behavior'].data_interfaces['Position'].spatial_series['cursor_pos']
go_cue_times = nwbfile.trials['go_cue_time'].data[:]
go_cue_times = go_cue_times[~np.isnan(go_cue_times)]  # Remove NaN values if any
window = [0.2, 0.5]  # 200 ms before to 500 ms after go cue

cursor_pos_chunks = get_windowed_pos_chunk(cursor_pos, go_cue_times, window)

for i, chunk in enumerate(cursor_pos_chunks):
    plt.plot(chunk[:,0],chunk[:,1])

Index(['start_time', 'stop_time', 'target_on_time', 'go_cue_time', 'target_id',
       'target_corners', 'target_dir', 'result'],
      dtype='object')


In [120]:
list_units_spkts = nwbfile.units[:]['spike_times']

In [141]:
spike_data = get_chunk_spikes_binned_windowed(list_units_spkts, go_cue_times, window, bin_size=0.01)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
ef plot_firing_rate_heatmap(firing_rates_2d, time_axis=None, bin_size=None,
                              figsize=(12, 8), cmap='viridis', 
                              vmin=None, vmax=None, title=None):
    """Plot firing rate heatmap for a 2D matrix.
    
    Args:
        firing_rates_2d: 2D array [n_units, n_timepoints] - spike counts or firing rates
        time_axis: Array of time values for x-axis. If None, uses indices.
        bin_size: Time bin size in seconds. If provided, converts counts to rates (Hz)
        figsize: Figure size tuple
        cmap: Colormap name ('viridis', 'hot', 'plasma', etc.)
        vmin, vmax: Color scale limits (None = auto)
        title: Plot title (None = default title)
    
    Returns:
        fig, ax: Figure and axis objects
    """
    # Convert to numpy array if needed
    firing_rates_2d = np.asarray(firing_rates_2d)
    n_units, n_timepoints = firing_rates_2d.shape
    
    # Convert spike counts to firing rates if bin_size provided
    if bin_size is not None:
        data_to_plot = firing_rates_2d / bin_size
        rate_label = 'Firing Rate (Hz)'
    else:
        data_to_plot = firing_rates_2d
        rate_label = 'Spike Count'
    
    # Create time axis if not provided
    if time_axis is None:
        time_axis = np.arange(n_timepoints)
        xlabel = 'Time Bin'
    else:
        time_axis = np.asarray(time_axis)
        xlabel = 'Time (s)'
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot heatmap
    im = ax.imshow(data_to_plot, 
                   aspect='auto',
                   cmap=cmap,
                   interpolation='nearest',
                   extent=[time_axis[0], time_axis[-1], n_units, 0],
                   vmin=vmin, vmax=vmax)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, label=rate_label)
    
    # Labels and title
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel('Unit Number', fontsize=12)
    
    if title is None:
        title = 'Neural Population Firing Rate'
    ax.set_title(title, fontsize=14)
    
    plt.tight_layout()
    return fig, ax

In [150]:
avg_spike_rates = np.mean(spike_data[0],axis=2)
avg_spike_rates.shape


(220, 18)